# <font color="steelblue">16. El IPC: inflación, tasas interanuales y euros reales</font>

*Unidad 3. Números índice y series cronológicas* · Sesión teórica 16 de 30

**Adaptación para el Grado en Periodismo**: Fernando Borrás Rocher (f.borras@umh.es)

**Basado en SIMUSTAT**, de Fernando Borrás, Federico Botella, Inés Hernández, M.ª Asunción Martínez Mayoral, Josep Moltó y Javier Morales.
Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`ipc_simulado.csv`, `ipc_ponderaciones.csv`, `salario_medio.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["ipc_simulado.csv", "ipc_ponderaciones.csv", "salario_medio.csv"]
PREGUNTAS = ["q_P16_1.json", "q_P16_2.json", "q_P16_3.json", "q_P16_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«Los salarios suben un 4 %»: buena noticia... salvo que los precios hayan subido un 6 %. Para saber si alguien gana o pierde poder de compra hay que **deflactar**: pasar de euros corrientes a **euros constantes**. Es una de las operaciones más útiles (y menos hechas) en información económica.

**Qué necesitas saber antes**: los cuadernos 14 y 15.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Leer una serie del IPC y calcular la inflación interanual y la media anual.
2. Entender el IPC general como media ponderada de sus grupos.
3. Deflactar una serie: pasar de euros corrientes a euros constantes.
4. Calcular la variación del poder adquisitivo.

## <font color='steelblue'>Contenidos</font>

1. La serie del IPC
2. El IPC general como media ponderada
3. Deflactar: euros corrientes y euros constantes
4. Poder adquisitivo

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. La serie del IPC</font>

In [ ]:
ipc = pd.read_csv("ipc_simulado.csv")
ipc["fecha"] = pd.to_datetime(ipc["fecha"])
ipc = ipc.set_index("fecha")
ipc.tail()

Es un IPC **simulado**, con base 2021 = 100 (como el IPC real del INE). La **inflación** que se publica cada mes es la **tasa interanual** del índice general.

In [ ]:
ipc["inflacion"] = ipc["general"].pct_change(12) * 100
ipc["inflacion"].plot(color="firebrick")
plt.axhline(0, color="grey", lw=0.8)
plt.title("Inflación interanual (IPC simulado)")
plt.ylabel("%")
plt.show()

También se publica la **inflación media anual**: la tasa entre la media del índice de un año y la del anterior. Con datos mensuales, `resample("YE")` agrupa por año.

In [ ]:
media_anual = ipc["general"].resample("YE").mean()
media_anual.index = media_anual.index.year
(media_anual.pct_change() * 100).round(1)

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P16_1.json")

## <font color='steelblue'>2. El IPC general como media ponderada</font>

In [ ]:
pond = pd.read_csv("ipc_ponderaciones.csv")
pond

El índice general es (aproximadamente) la media de los índices de cada grupo, ponderada por el peso de ese grupo en el gasto de los hogares:

In [ ]:
grupos = pond["grupo"].tolist()
reconstruido = (ipc[grupos] * pond.set_index("grupo")["ponderacion"]).sum(axis=1)
pd.DataFrame({"general": ipc["general"], "reconstruido": reconstruido.round(3)}).tail(3)

Coinciden porque así se generó el índice simulado. En el IPC real coinciden de forma aproximada, porque el INE encadena las ponderaciones de cada año. Comparemos cómo ha subido cada grupo:

In [ ]:
(ipc[grupos].iloc[-1] - 100).sort_values().plot(kind="barh", color="steelblue")
plt.xlabel("Variación desde la media de 2021 (%)")
plt.title("¿Qué ha subido más desde 2021? (IPC simulado)")
plt.show()

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P16_2.json")

## <font color='steelblue'>3. Deflactar: euros corrientes y euros constantes</font>

Para comparar euros de años distintos hay que quitar el efecto de los precios. Si el IPC es $I_t$ (base 100), el **valor real** en euros del año base es:

$$\text{valor real}_t = \frac{\text{valor nominal}_t}{I_t} \times 100$$

In [ ]:
sal = pd.read_csv("salario_medio.csv").set_index("anio")
sal["ipc_medio"] = media_anual.round(2)
sal["salario_real_2021"] = (sal["salario_medio_nominal"] / sal["ipc_medio"] * 100).round(0)
sal

In [ ]:
sal[["salario_medio_nominal", "salario_real_2021"]].plot(marker="o")
plt.title("Salario medio nominal y real, en euros de 2021 (datos simulados)")
plt.ylabel("€")
plt.show()

**Cómo leerlo**: el salario nominal sube todos los años. El real, no: entre 2020 y 2023 los precios subieron más que los sueldos y el poder de compra **cayó**, aunque la nómina fuera mayor.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escribe el número y pulsa Enter:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P16_3.json")

## <font color='steelblue'>4. Poder adquisitivo</font>

La variación del **poder adquisitivo** es la tasa de variación del salario real. Una aproximación rápida, útil para tasas pequeñas, es: variación del salario − inflación.

In [ ]:
sal["var_nominal"] = (sal["salario_medio_nominal"].pct_change() * 100).round(1)
sal["inflacion_media"] = (sal["ipc_medio"].pct_change() * 100).round(1)
sal["var_real"] = (sal["salario_real_2021"].pct_change() * 100).round(1)
sal[["var_nominal", "inflacion_media", "var_real"]]

Un titular honesto con estos datos sería del tipo: «Los salarios suben un X %, pero pierden poder de compra por tercer año» (ajústalo a lo que digan las cifras).

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P16_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 1)*

¿En qué mes se alcanzó la inflación interanual más alta de la serie y cuánto fue?

<details><summary>Ver solución</summary>

```python
ipc["inflacion"].idxmax(), round(ipc["inflacion"].max(), 1)
```
</details>

### **Ejercicio 2** *(sección 2)*

Calcula la tasa interanual del grupo «vivienda» y compárala en un gráfico con la del índice general.

<details><summary>Ver solución</summary>

```python
(ipc[["general", "vivienda"]].pct_change(12) * 100).plot()
plt.ylabel("%")
plt.show()
```
</details>

### **Ejercicio 3** *(sección 3)*

Expresa el salario real en euros de 2015 en lugar de euros de 2021.

<details><summary>Ver solución</summary>

```python
(sal["salario_medio_nominal"] / sal["ipc_medio"] * sal.loc[2015, "ipc_medio"]).round(0)
```
</details>

### **Ejercicio 4** *(sección 4)*

¿Entre 2015 y 2025 el salario real acumulado subió o bajó? ¿Cuánto?

<details><summary>Ver solución</summary>

```python
round((sal.loc[2025, "salario_real_2021"] / sal.loc[2015, "salario_real_2021"] - 1) * 100, 1)
```
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- La **inflación** que se publica es la **tasa interanual** del IPC general; la **media anual** compara medias de años.
- El IPC general es una **media ponderada** de los índices de sus grupos.
- **Deflactar**: valor real = valor nominal / IPC × 100 (en euros del año base).
- El **poder adquisitivo** cambia según la variación del salario real ≈ subida salarial − inflación.
- Una subida en euros corrientes puede ser una **pérdida** en euros constantes.

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **IPC** | Índice de Precios de Consumo: mide la evolución de los precios que pagan los hogares. |
| **Inflación** | Aumento generalizado de los precios; se mide con la tasa interanual del IPC. |
| **Deflactar** | Eliminar el efecto de los precios para expresar valores en euros constantes. |
| **Euros corrientes** | Euros de cada año, sin corregir. |
| **Euros constantes** | Euros de un año de referencia. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `resample` da error | El índice no es de fechas | Convierte con `pd.to_datetime` y usa `set_index` |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 17 aprenderás a cambiar la base de un índice y a enlazar series.

**Siguiente cuaderno**: [17. Cambio de base y enlace de series](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P17_cambio_de_base_y_enlace.ipynb)